For all assembled datasets by year and country, we will clean them in two ways:
(1) only extracting BPM and energy,
(2) normalizing everything to 100 instead of 1

In [1]:
import os, glob, re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt

In [176]:
FILE_PATH = '/Users/kaiyu/Downloads/y3s1 BT4241' # replace with your file path
years = list(range(2004, 2014))
country_charts_dict = {
                        "SGP": "987FM Year End Charts", # Singapore
                        'KOR': "Melon Year End Charts", # South Korea
                        # 'NZL': "Official Aotearoa Year End Charts", # New Zealand
                        'BRA': "Mais Tocadas Year End Charts", # Brazil
                        }
columns_to_read = ['Tempo', 'Danceability', 'Energy', 'Valence', 'Loudness']

In [177]:
yu_df = pd.read_csv('unemployment-rate-for-young-people.csv')
countries = ['KOR', 'BRA', 'SGP']
yu_df = yu_df.loc[(yu_df['Code'].isin(countries))].reset_index()
yu_df = yu_df.drop('index', axis=1)
yu_df

,Entity,Code,Year,"Unemployment, youth total (% of total labor force ages 15-24) (modeled ILO estimate)"
0,Brazil,BRA,1991,12.763
1,Brazil,BRA,1992,13.119
2,Brazil,BRA,1993,11.406
3,Brazil,BRA,1994,12.353
4,Brazil,BRA,1995,13.261
...,...,...,...,...
97,South Korea,KOR,2020,10.143
98,South Korea,KOR,2021,8.055
99,South Korea,KOR,2022,6.638
100,South Korea,KOR,2023,5.405


In [ ]:
years_str = list(map(str, list(range(2004, 2014))))
gdp_stats_df = pd.read_csv('gdp_growth.csv')
countries = ['KOR', 'BRA', 'SGP']
gdp_stats_df = gdp_stats_df.loc[(gdp_stats_df['Country Code'].isin(countries))].reset_index()
gdp_stats_df = gdp_stats_df.drop('index', axis=1)
gdp_stats_df

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,Unnamed: 69
0,Brazil,BRA,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,8.600000,6.600000,0.600000,3.400000,2.400000,...,-3.275917,1.322869,1.783667,1.220778,-3.276759,4.762604,3.016694,3.241655,3.395866,NaN
1,"Korea, Rep.",KOR,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,6.935993,3.895273,9.020568,9.473825,7.318434,...,2.946882,3.159636,2.907404,2.243978,-0.709415,4.304735,2.612672,1.356733,NaN,NaN
2,Singapore,SGP,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,8.137530,7.553444,10.040173,-3.103168,7.834260,...,3.747578,4.476899,3.451976,1.308041,-3.814709,9.756804,4.108000,1.821407,4.388024,NaN


In [49]:
val = gdp_stats_df.at[0, '2016'] # test of Brazilian GDP Growth in 2016
print(val)

-3.27591690782192


In [151]:
rate = yu_df.loc[(yu_df['Code'] == 'SGP') & (yu_df['Year'] == 2000),
                'Unemployment, youth total (% of total labor force ages 15-24) (modeled ILO estimate)'].values[0] /100

print(rate)

0.06827


In [178]:
chart_panel_df = pd.DataFrame(columns=['Country', 'Treat_Grp', 'Year', 'post_recession', 'BPM', 'Danceability', 'Energy', 'Happy', 'Loudness'])
for country_code, chart_name in country_charts_dict.items():
    print(country_code)
    curr_country_dir = os.path.join(FILE_PATH, chart_name)
    curr_country_files = os.listdir(curr_country_dir)
    curr_country_files.sort()
    country_col_in_gdp_df = yu_df.index[yu_df['Code'] == country_code].tolist()[0]
    if country_code == 'SGP' or country_code == 'KOR':
        curr_country_files.remove('.DS_Store')
    for filename in curr_country_files:
        year = re.split(r'[_.]', filename)[0] if country_code == 'BRA' else re.split(r'[_.]', filename)[-2]
        curr_year_csv = pd.read_csv(os.path.join(curr_country_dir, filename), usecols=columns_to_read)
        median_audio_features = curr_year_csv.mean()
        post_recession = 0 if int(year) < 2009 else 1
        youth_unemployment = yu_df.loc[(yu_df['Code'] == country_code) & (yu_df['Year'] == int(year)),
                                            'Unemployment, youth total (% of total labor force ages 15-24) (modeled ILO estimate)'].values[0] / 100
        new_row_df = pd.DataFrame({
                'Country': [country_code], 
                'Treat_Grp': [0],
                'Year': [int(year)],
                'youth_unemployment': youth_unemployment, 
                'post_recession': [post_recession], 
                'BPM': [median_audio_features['Tempo']], 
                'Danceability': [median_audio_features['Danceability']], 
                'Energy': [median_audio_features['Energy']],
                'Happy': [median_audio_features['Valence']],
                'Loudness': [median_audio_features['Loudness']]
            })
        chart_panel_df = pd.concat([chart_panel_df, new_row_df], ignore_index=True)

SGP
KOR
BRA


/var/folders/yd/y56txp5d0t1_5b8nszp56my80000gn/T/ipykernel_55290/3054951946.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  chart_panel_df = pd.concat([chart_panel_df, new_row_df], ignore_index=True)


In [179]:
chart_panel_df.shape # check for the correct shape (4*10 observations of 9 features)

(42, 10)

In [180]:
# concatenate and save
aus_usa_df = pd.read_csv('chart_panel.csv', index_col=0)
full_chart_panel_df = pd.concat([chart_panel_df, aus_usa_df], ignore_index=True)
full_chart_panel_df.shape

(70, 10)

In [158]:
full_chart_panel_df.dtypes

Country                object
Treat_Grp              object
Year                   object
post_recession         object
BPM                   float64
Danceability          float64
Energy                float64
Happy                 float64
Loudness              float64
youth_unemployment    float64
dtype: object

In [181]:
full_chart_panel_df = full_chart_panel_df.sort_values(['Country', 'Year'])
full_chart_panel_df.to_csv("combined_panel_chart_data.csv")

In [189]:
import statsmodels.formula.api as smf

model = smf.ols(
    formula="Energy ~ Loudness + youth_unemployment + C(Year)",
    data=full_chart_panel_df
).fit(cov_type="HC1")  # HC1 = robust (heteroskedasticity-consistent)

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 Energy   R-squared:                       0.748
Model:                            OLS   Adj. R-squared:                  0.678
Method:                 Least Squares   F-statistic:                     10.47
Date:                Tue, 25 Nov 2025   Prob (F-statistic):           4.12e-11
Time:                        23:27:53   Log-Likelihood:                 167.85
No. Observations:                  70   AIC:                            -303.7
Df Residuals:                      54   BIC:                            -267.7
Df Model:                          15                                         
Covariance Type:                  HC1                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              1.1228      0

In [190]:
model.pvalues

Intercept             9.068132e-112
C(Year)[T.2001]        2.703545e-02
C(Year)[T.2002]        1.189776e-02
C(Year)[T.2003]        1.321641e-03
C(Year)[T.2004]        1.898747e-04
C(Year)[T.2005]        3.204449e-03
C(Year)[T.2006]        2.554317e-04
C(Year)[T.2007]        2.638856e-03
C(Year)[T.2008]        5.022334e-03
C(Year)[T.2009]        1.844568e-04
C(Year)[T.2010]        4.666064e-03
C(Year)[T.2011]        3.197393e-03
C(Year)[T.2012]        2.846462e-03
C(Year)[T.2013]        2.603151e-04
Loudness               8.671707e-17
youth_unemployment     1.263712e-06
dtype: float64